# 04 · Title chore and scan churn — what the background work costs the GUI

The chore that generates row titles and the scan that discovers sessions both
run on a timer, both call out to an LLM, and both feed the sidebar. This
notebook asks what they cost and how much of it is **re-work on rows that did
not change**.

Two known shapes to look for:

* **Fake recency.** A scanner that stamps every row it finds with the *database
  file's* mtime makes the whole set look freshly modified every time anything
  touches that file, so a re-sort and a re-render follow work that changed
  nothing.
* **429 back-pressure.** The LLM endpoint rate-limits under bursts. A heuristic
  fallback must never be persisted over a 429 — that writes a wrong title
  durably and the next run sees a titled row and skips it.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) != "notebooks" else os.getcwd()))
sys.path.insert(0, os.path.abspath("."))
import ytrace_helpers as H

WINDOW = os.environ.get("YGG_NOTEBOOK_WINDOW", "30m")
HOST = H.GUI_HOST
H.describe_source(HOST, WINDOW)

In [ ]:
title_gen  = [r for r in H.tail(HOST, since=WINDOW, category="copy_generation") if r.get("name") == "title"]
title_probe= H.tail(HOST, since=WINDOW, category="title")
scans      = H.tail(HOST, since=WINDOW, category="background")
chore      = [r for r in H.tail(HOST, since=WINDOW, category="daemon") if "copy" in str(r.get("name"))]
sidebar    = [r for r in H.tail(HOST, since=WINDOW, category="sidebar") if r.get("name") == "merge_rows"]

for label, rows in [("copy_generation/title", title_gen), ("title/*", title_probe),
                    ("background/*", scans), ("daemon copy chore", chore),
                    ("sidebar/merge_rows", sidebar)]:
    print(f"{label:22} {len(rows):6d}")

In [ ]:
# Wall-clock cost of one title generation. This is a WALL span: it is latency,
# and it is dominated by the LLM round trip.
gen_ms = [r["duration_ms"] for r in title_gen if isinstance(r.get("duration_ms"), (int, float))]
st = H.percentiles(gen_ms)
print("copy_generation/title (wall ms):", st)
if st["n"]:
    span_min = (max(r["ts_ms"] for r in title_gen) - min(r["ts_ms"] for r in title_gen)) / 60000 or 1
    print(f"rate: {st['n']/span_min:.2f} generations/min over {span_min:.1f} min")
    print(f"occupancy: {sum(gen_ms)/1000:.0f} s of generation inside a {span_min*60:.0f} s window "
          f"= {sum(gen_ms)/(span_min*60000):.2f} concurrent generations")
    print("series:", H.sparkline(gen_ms))

In [ ]:
# Scan cost and how many rows each scan claims to have touched.
scan_ms = [r["duration_ms"] for r in scans if isinstance(r.get("duration_ms"), (int, float))]
print("background scan (wall ms):", H.percentiles(scan_ms))

marked, scanned = [], []
for r in scans + chore:
    p = r.get("payload") or {}
    if not isinstance(p, dict):
        continue
    for key in ("modified", "changed", "marked_modified", "rows_modified", "updated"):
        if isinstance(p.get(key), (int, float)):
            marked.append(float(p[key])); break
    for key in ("rows", "scanned", "total", "sessions", "count"):
        if isinstance(p.get(key), (int, float)):
            scanned.append(float(p[key])); break

print("rows a scan reports as MODIFIED :", H.percentiles(marked) if marked else "(payload carries no such field)")
print("rows a scan reports as SCANNED  :", H.percentiles(scanned) if scanned else "(payload carries no such field)")
churn = None
if marked and scanned:
    churn = H.percentiles(marked).get("p50", 0) / max(H.percentiles(scanned).get("p50", 1), 1)
    print(f"\nchurn ratio (modified/scanned, p50) = {churn:.2%}")
    print("=> " + ("EVERY row looks modified on every tick — the fake-recency shape"
                   if churn > 0.9 else "a minority of rows change per tick, as expected"))

In [ ]:
# 429s and heuristic fallbacks. A 429 that produced a persisted title is the
# defect; a 429 that stopped the tick is the design working.
rate_limited, fallbacks = 0, 0
for r in title_gen + title_probe + chore:
    blob = str(r.get("payload"))
    if "429" in blob or "rate_limit" in blob or "rate limit" in blob.lower():
        rate_limited += 1
    if "heuristic" in blob.lower() or "fallback" in blob.lower():
        fallbacks += 1
print(f"records mentioning a 429 / rate limit : {rate_limited}")
print(f"records mentioning a heuristic fallback: {fallbacks}")

llm = [r for r in title_probe if r.get("name") in ("llm_rescue", "generation", "resolve_attempt")]
print(f"title/* LLM-path probes                : {len(llm)}")
if not title_probe:
    print("\nNOTE: title/* is registered but silent; the title work that ran emitted "
          "under copy_generation/title. See docs/observability.md §4.4.")

In [ ]:
# Sidebar re-merge is the GUI-side consequence: every scan that marks rows dirty
# lands here, and this cost is paid on the GUI host.
merge_ms = [r["duration_ms"] for r in sidebar if isinstance(r.get("duration_ms"), (int, float))]
mst = H.percentiles(merge_ms)
print("sidebar/merge_rows (wall ms):", mst)
if mst["n"]:
    span_min = (max(r["ts_ms"] for r in sidebar) - min(r["ts_ms"] for r in sidebar)) / 60000 or 1
    rate = mst["n"] / span_min
    occupancy = sum(merge_ms) / (span_min * 60000)
    print(f"rate {rate:.1f} merges/min; occupancy {occupancy:.3f} cores of continuous merging")

In [ ]:
TITLE_GEN_WARN_MS, TITLE_GEN_FAIL_MS = 5_000.0, 12_000.0
MERGE_OCCUPANCY_WARN = 0.05
CHURN_WARN = 0.90

v = H.Verdict("Title chore and scan churn")
v.check("title generation p50 (wall)", st.get("p50"), f"<= {TITLE_GEN_WARN_MS/1000:.0f} s",
        warn_over=TITLE_GEN_WARN_MS, fail_over=TITLE_GEN_FAIL_MS, detail=f"n={st.get('n', 0)}")
v.check("rate-limited records", float(rate_limited), "0", warn_over=0, fail_over=5)
v.check("heuristic fallback mentions", float(fallbacks), "0 persisted over a 429", warn_over=0, fail_over=5)
v.check("scan churn ratio", churn, f"<= {CHURN_WARN:.0%} of rows dirty per tick",
        warn_over=CHURN_WARN, fail_over=0.99,
        detail="every row dirty every tick is the fake-recency shape")
v.check("sidebar merge occupancy", (sum(merge_ms)/(((max(r['ts_ms'] for r in sidebar)-min(r['ts_ms'] for r in sidebar))/60000 or 1)*60000)) if merge_ms else None,
        f"<= {MERGE_OCCUPANCY_WARN} cores", warn_over=MERGE_OCCUPANCY_WARN, fail_over=0.25)
v.show()